<a href="https://colab.research.google.com/github/Bovi-analytics/Ansci4040FinalClassProject/blob/main/ReidParlorCalving.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cow re-ID with real farm IDs: parlor pool → calving pens

**Goal:** name the cows seen in the calving pens with their farm ID, from what they look like.

**Where the IDs come from** (no labelling by hand):
- The BoviSync parlor report for one day (`BoviSyncRecords/Nov17Parlor.xlsx`) gives every milking: cow, attach time and stall.
- A *turn* is one side of the parlor filling up. Its cows share one attach minute.
- Every cow is milked 3 times a day, with different neighbours each time.
- For each cow, the code picks the one track per milking that holds her (by turn) and that look most alike. A cow is named only when that choice is clear.
- This gives the **parlor pool**: crops with real farm IDs.

**The test:** the calving-pen cows of the days before (Nov 11–12) are named against that pool. The cows that calved those days (`Nov11To15Calvings.xlsx`) should come out. Chance level is printed next to the score.

| Cell | Step | Model | Time (T4, about) |
|---|---|---|---|
| 3 | Parlor clock: each 8-h part's start, the milkings in it, one check picture per milking | AVI chunk walk | a few min per video |
| 4 | Parlor crops with farm IDs: the parlor group's own named crops (default), or SAM cut-outs at their hand-timed seconds | none / SAM 2.1 | seconds / ~10 min (GPU) |
| 5 | Track + crops, calving pens, the 2 busiest 15-min windows per video | YOLO26n + ByteTrack, SAM 2.1 | ~20 min per window |
| 6 | 8-band fingerprints of every crop | DINOv3 ViT-S/16 | 5–10 min per folder |
| 7 | Load the parlor crops; with 2+ milkings also name every parlor cow → `Names/ParlorIds.csv`, `Names/Cows/<CowId>.jpg` | none (report + fingerprints) | minutes |
| 8 | Name the calving-pen cows → `Names/CalvingMatches.csv`, `Names/CalvingSummary.csv` | none | minutes |

Every cell continues where it stopped when rerun, and skips what is done. Outputs: `ReidPipelineOutputs/<Run>/`. Needs a T4 GPU for cells 4–6 and `HF_TOKEN` in Colab Secrets. Cells 3, 7 and 8 also run on a CPU runtime.

In [ ]:
# 1. Setup: mount Drive, get the code from GitHub, install the libraries.
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/Bovi-analytics/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"
if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
import json, re
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
from CalvingPipeline.Paths import BoviSyncDir, FindVideo, GroupDir, ReidOutputRoot
from CalvingPipeline.AviFrames import AviActivity
from CalvingPipeline.Reid.Frames import HeaderFps, LocalCopy
from CalvingPipeline.Reid.Segments import TrackAndCrop
from CalvingPipeline.Reid.Bands import EmbedBands
from CalvingPipeline.Reid.PenArea import PenAreaFor
from CalvingPipeline.Reid.Validate import LoadVideo
from CalvingPipeline.Reid.ParlorIds import (ActiveWindows, CowSheets, LoadMany, MatchCalving, MilkingFrames, NameParlor,
                                            PartOffsets, RankCalvedCows, ReadCalvings, RecordedMinutes, ReadParlorReport,
                                            SessionWindows)
from CalvingPipeline.Reid.ParlorStamps import CropStamps, LabelStamps, ReadStamps, StampCheck, StampSheet, TeamPool
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (fine for cells 3, 7, 8; cells 4-6 need a T4)")
try:
    HfToken = userdata.get("HF_TOKEN")   # from Colab Secrets; never paste the token here
except Exception:
    HfToken = None
print("HF_TOKEN:", "found" if HfToken else "NOT SET - cell 6 (DINOv3) will stop; add it in Colab Secrets")

In [ ]:
# 2. Settings: the only cell to edit.
Run = "Parlor17NovCalving11Nov"          # output folder: ReidPipelineOutputs/<Run>
Year = 2026

# --- Parlor: one day of entry-parlor video + that day's BoviSync parlor report ---
ParlorVideos = ["EntryParlor17Nov.avi", "EntryParlor0217Nov.avi", "EntryParlor0317Nov.avi"]   # parts 1-3 of 17 Nov
ParlorReport = BoviSyncDir / "Nov17Parlor.xlsx"
ParlorStarts = None                      # None = the parts follow each other from 00:00 (the camera cuts a file every 8 h);
                                         # or one "HH:MM" per part, e.g. ["00:00", "08:00", "16:00"]
# Where the parlor cows come from:
#   "Team"   = the parlor group's own named crops: ParlorIDMatch/Nov17M1CropCowTeamVaca.zip (one crop per hand-timed cow,
#              milking 1, right side) + BoviSyncRecords/CleanedData/CleanedNov17MilkingRight.xlsx (farm ID per place in
#              the order). Their IDs and ours from the stalls agree on all 72 named cows.
#   "Stamps" = a hand-timed list: the second each cow walks into one side (the parlor group's ParlorIDMatch/TimeSecond.xlsx,
#              milking 1, right side = stalls 1-28). No detector needed. Our own list for the other side works the same.
#   "Tracks" = YOLO + ByteTrack in the parlor (does not find the cows in this camera view: kept for other cameras)
ParlorSource = "Team"
TeamZip = GroupDir / "ParlorIDMatch" / "Nov17M1CropCowTeamVaca.zip"
TeamOrder = BoviSyncDir / "CleanedData" / "CleanedNov17MilkingRight.xlsx"
StampFile = GroupDir / "ParlorIDMatch" / "TimeSecond.xlsx"
StampVideo, StampSession, StampSide = "EntryParlor17Nov.avi", 1, "A"   # the video the times are in; milking; side "A" = stalls 1-28
FirstCowStall = 1                        # the first cow in walks to stall 1 (the parlor group's rule)
StampPoint = "Motion"                    # "Motion" = at each time, the cow moving on the stalls 1-28 side (right third of the
                                         # picture, away from the pit) is cut out; or a fixed (x, y) in the video's pixels;
                                         # None = only show StampCheck.jpg
StampOffsets = (1.0, 2.0, 3.0)           # crops this many seconds after each time
Milkings = [2]                           # "Tracks" only: which milkings to track: [2] = midday only (daylight, ~1 h on a T4);
                                         # [1, 2, 3] = all three (also names every parlor cow from the report, cell 7)
ParlorDetModel = "yolo26m.pt"            # medium YOLO: cows packed shoulder to shoulder

# --- Calving pens: the days to name, and the calving list ---
CalvingVideos = ["CalvingPen1Nov11Cow1.avi", "CalvingPen2Nov11Cow1.avi", "CalvingPen3Nov11Cow1.avi",
                 "CalvingPen2Nov12Cow2.avi", "CalvingPen3Nov12Cow2.avi", "CalvingPen3Nov12Cow4.avi"]
CalvingReport = BoviSyncDir / "Nov11To15Calvings.xlsx"
WindowsPerVideo, WindowMinutes = 2, 15   # the busiest stretches of each recording (a cow moving)
ExpectedDays = 1                         # a cow filmed on day D should be one that calved D-1 .. D+1

# --- Tracking and crops (cells 4-5) ---
TargetFps, EverySeconds = 5, 15          # frames looked at per second; a crop of every cow every 15 s
ImageSize, Confidence = 1024, 0.25
GateArea = 10000                         # smallest usable mask, for 1920x1080 (scaled to each video's size)

# --- Naming parlor cows (cell 7) ---
MinScore = 0.75                          # her tracks from different milkings must look at least this alike
MinMargin = 0.03                         # ... and clearly more alike than any other choice
DeleteLocalCopies = True                 # free the Colab disk after each video

RunDir = ReidOutputRoot / Run
ParlorDir, CalvingDir, NamesDir = RunDir / "Parlor", RunDir / "Calving", RunDir / "Names"
for D in (ParlorDir, CalvingDir, NamesDir):
    D.mkdir(parents=True, exist_ok=True)
Track = dict(TargetFps=TargetFps, EverySeconds=EverySeconds, ImageSize=ImageSize, Confidence=Confidence)
Report = ReadParlorReport(ParlorReport)
Calvings = ReadCalvings(CalvingReport)
def VideoDate(Name):
    M = re.search(r"CalvingPen\d+([A-Z][a-z]{2})(\d{1,2})", Name)
    return pd.Timestamp(f"{M.group(2)} {M.group(1)} {Year}")
def Drop(Video):
    if DeleteLocalCopies:
        (Path("/content/ReidVideos") / Path(Video).name).unlink(missing_ok=True)
print(f"Parlor report: {len(Report)} milkings of {Report.CowId.nunique()} cows in {Report.Session.nunique()} sessions; "
      f"calving list: {len(Calvings)} calvings. Outputs: {RunDir}")

In [ ]:
# 3. Parlor clock: when each part starts and where the milkings of the report fall in it.
#    The entry-parlor camera records all the time and cuts a file every 8 h, so part 1 = 00:00-08:00, part 2 =
#    08:00-16:00, part 3 = 16:00-24:00. Check Parlor/<video>Milkings.jpg: cows in the stalls in every picture.
Clock, Minutes = {}, []
for V in ParlorVideos:
    ActPath = ParlorDir / f"{Path(V).stem}Activity.csv"
    if ActPath.exists():
        Activity = pd.read_csv(ActPath)
    else:
        Local = LocalCopy(FindVideo(V))
        Activity = AviActivity(Local, TicksPerSecond=HeaderFps(Local))
        Activity.to_csv(ActPath, index=False)
    Minutes.append(round(RecordedMinutes(Activity)))
for V, Offset, M in zip(ParlorVideos, PartOffsets(Minutes, ParlorStarts), Minutes):
    Windows = SessionWindows(Report, Offset, M)
    Clock[V] = dict(OffsetMinutes=Offset, VideoMinutes=M,
                    Windows=Windows[["Session", "StartSeconds", "EndSeconds"]].to_dict("records"))
    Check = ParlorDir / f"{Path(V).stem}Milkings.jpg"
    if Clock[V]["Windows"] and not Check.exists():
        MilkingFrames(LocalCopy(FindVideo(V)), Clock[V]["Windows"], Check)
(ParlorDir / "Clock.json").write_text(json.dumps(Clock, indent=1))
display(pd.DataFrame([{"Video": V, "StartsAt": f"{C['OffsetMinutes'] // 60:02d}:{C['OffsetMinutes'] % 60:02d}",
                       "Hours": round(C["VideoMinutes"] / 60, 1), "Milkings": [W["Session"] for W in C["Windows"]],
                       "Windows": [f"{W['StartSeconds'] / 3600:.2f}-{W['EndSeconds'] / 3600:.2f} h" for W in C["Windows"]]}
                      for V, C in Clock.items()]))
for V in ParlorVideos:
    if (ParlorDir / f"{Path(V).stem}Milkings.jpg").exists():
        display(Image(str(ParlorDir / f"{Path(V).stem}Milkings.jpg"), width=1200))

In [ ]:
# 4. Parlor crops with farm IDs. "Stamps": each hand-timed cow gets the cow of her stall in the report, and SAM cuts out
#    the cow under StampPoint a few seconds after her time. First run (StampPoint = None): look at StampCheck.jpg, pick a
#    spot every cow walks over (x, y from the grid), set StampPoint in cell 2, rerun cells 2 and 4.
StampDir = ParlorDir / "Stamps"
PoolDir = ParlorDir / ("Team" if ParlorSource == "Team" else "Stamps")
if ParlorSource == "Team":
    TeamPool(TeamZip, TeamOrder, PoolDir)
    Named = pd.read_csv(PoolDir / "Stamps.csv").dropna(subset=["CowId"])
    Days = sorted({VideoDate(V) for V in CalvingVideos})
    Around = set(Calvings.CowId[Calvings.Date.between(Days[0] - pd.Timedelta(days=ExpectedDays), Days[-1] + pd.Timedelta(days=ExpectedDays))])
    print(f"of them calved within {ExpectedDays} day(s) of the calving videos: {sorted(set(Named.CowId.astype(int)) & Around)}")
elif ParlorSource == "Stamps":
    StampDir.mkdir(parents=True, exist_ok=True)
    Stamps = LabelStamps(ReadStamps(StampFile), Report, Session=StampSession, Side=StampSide, FirstCowStall=FirstCowStall)
    Days = sorted({VideoDate(V) for V in CalvingVideos})
    Around = set(Calvings.CowId[Calvings.Date.between(Days[0] - pd.Timedelta(days=ExpectedDays), Days[-1] + pd.Timedelta(days=ExpectedDays))])
    print(f"{len(Stamps)} timed cows, {Stamps.CowId.notna().sum()} with a farm ID; of them calved within {ExpectedDays} day(s) "
          f"of the calving videos: {sorted(set(Stamps.CowId.dropna().astype(int)) & Around)}")
    if StampPoint is None:
        StampCheck(FindVideo(StampVideo), Stamps, StampDir / "StampCheck.jpg")
        display(Image(str(StampDir / "StampCheck.jpg"), width=1200))
        print("Rows = the first 3 cows, columns = 2 s before, at, 2 s and 4 s after her time. Set StampPoint in cell 2.")
    else:
        CropStamps(FindVideo(StampVideo), Stamps, StampDir, StampPoint, Offsets=StampOffsets)
        display(Image(str(StampSheet(StampDir)), width=1200))
        print("Each row should be one cow (label: report turn, stall, CowId), the whole cow, not a patch or the person.")
    Drop(StampVideo)
else:
    for V in ParlorVideos:
        for W in [W for W in Clock[V]["Windows"] if W["Session"] in Milkings]:
            print(f"\n=== {V}, milking {W['Session']} ===")
            TrackAndCrop(FindVideo(V), ParlorDir / f"{Path(V).stem}S{W['Session']}", StartSeconds=float(W["StartSeconds"]),
                         EndSeconds=float(W["EndSeconds"]), DetModel=ParlorDetModel, **Track)
        Drop(V)

In [ ]:
# 5. Calving pens: track + crops in the busiest windows of each recording (folders <video>Win1, Win2). Rerun to continue.
for V in CalvingVideos:
    Stem, ActPath = Path(V).stem, CalvingDir / f"{Path(V).stem}Activity.csv"
    if ActPath.exists():
        Activity = pd.read_csv(ActPath)
    else:
        Local = LocalCopy(FindVideo(V))
        Activity = AviActivity(Local, TicksPerSecond=HeaderFps(Local))
        Activity.to_csv(ActPath, index=False)
    for K, (Start, End) in enumerate(ActiveWindows(Activity, WindowsPerVideo, WindowMinutes), 1):
        print(f"\n=== {V}, window {K}: {Start / 3600:.2f}-{End / 3600:.2f} h ===")
        TrackAndCrop(FindVideo(V), CalvingDir / f"{Stem}Win{K}", StartSeconds=float(Start), EndSeconds=float(End),
                     DetModel="yolo26n.pt", **Track)
    Drop(V)

In [ ]:
# 6. 8-band fingerprints (DINOv3 ViT-S/16) of every crop in every folder. Skipped when up to date.
ParlorFolders = ([PoolDir] if ParlorSource in ("Team", "Stamps") else
                 [ParlorDir / f"{Path(V).stem}S{W['Session']}" for V in ParlorVideos for W in Clock[V]["Windows"]
                  if W["Session"] in Milkings])
Folders = [D for D in ParlorFolders + sorted(CalvingDir.iterdir()) if (D / "Crops.csv").exists()]
for D in Folders:
    EmbedBands(D, Token=HfToken)

In [ ]:
# 7. Parlor crops of the chosen milkings. With 2+ milkings, also name every parlor cow from the report: per cow,
#    one track per milking that holds her and looks most alike (check Names/Cows/<CowId>.jpg: rows = same cow).
if ParlorSource in ("Team", "Stamps"):
    Use = [(PoolDir, Clock[StampVideo]["OffsetMinutes"])]
else:
    Use = [(ParlorDir / f"{Path(V).stem}S{W['Session']}", Clock[V]["OffsetMinutes"]) for V in ParlorVideos
           for W in Clock[V]["Windows"] if W["Session"] in Milkings]
Use = [(D, O) for D, O in Use if (D / "Bands.npz").exists()]
PCrops, PBands = LoadMany([D for D, _ in Use], [O for _, O in Use], GateArea=GateArea)   # TimeSeconds = seconds of the day
Keep = (PCrops.Gated & ~PCrops.NonCow).to_numpy()
PCrops, PBands = PCrops[Keep].reset_index(drop=True), PBands[Keep]
print(f"Parlor: {PCrops.TrackId.nunique()} tracks, {len(PCrops)} usable crops in milking(s) {sorted(Milkings)}")
Ids = None
if ParlorSource in ("Team", "Stamps"):      # the hand-timed cows already carry their farm ID
    Named = pd.read_csv(PoolDir / "Stamps.csv").dropna(subset=["CowId"])
    Ids = pd.DataFrame({"TrackId": Named.StampId.astype(int), "CowId": Named.CowId.astype(int), "Session": StampSession})
    Ids = Ids[Ids.TrackId.isin(PCrops.TrackId)]
    print(f"{Ids.CowId.nunique()} parlor cows with a farm ID and usable crops")
elif len(set(PCrops.Folder)) >= 2:
    Ids = NameParlor(PCrops, PBands, Report, 0, Width=int(PCrops.Width.iloc[0]), OutputDir=NamesDir,
                     MinScore=MinScore, MinMargin=MinMargin)
    CowSheets(PCrops, Ids, NamesDir)
    print(f"{Ids.CowId.nunique()} of {Report.CowId.nunique()} parlor cows named ({len(Ids)} tracks)")
    display(Ids.groupby("CowId").agg(Milkings=("Session", "size"), Score=("Score", "first"), Margin=("Margin", "first"))
               .sort_values("Margin", ascending=False).head(20))

In [ ]:
# 8. Name the calving-pen cows. One milking: each calving-pen track is ranked against the cows that calved within
#    ExpectedDays of its video, using only the parlor tracks that could be each cow (her stall side, her turn's time).
#    Which picture side holds stalls 1-28 is chosen once for all videos (the clearer answers). Margin 0 = a tie: two
#    calved cows were in the same turn; a second milking separates them. Two or more milkings: matched against the
#    named parlor pool (ExpectedShare vs Chance).
Pens = []
for D in sorted(D for D in CalvingDir.iterdir() if (D / "Bands.npz").exists()):
    Calved = set(Calvings.CowId[(Calvings.Date - VideoDate(D.name)).abs() <= pd.Timedelta(days=ExpectedDays)])
    C, B, _ = LoadVideo(D, PenArea=PenAreaFor(D.name), GateArea=GateArea)
    Ok = (C.Gated & ~C.NonCow).to_numpy()
    if Ok.any():
        Pens.append((D.name, C[Ok].reset_index(drop=True), B[Ok], Calved))
def Ranked(Layout):
    Out = []
    for Name, C, B, Calved in Pens:
        if Ids is None:
            M = RankCalvedCows(C, B, PCrops, PBands, Report, Calved, SideOnLeft=Layout)
        else:
            Named = PCrops.TrackId.isin(Ids.TrackId).to_numpy()
            Pool = PCrops[Named].assign(CowId=PCrops.TrackId[Named].map(Ids.set_index("TrackId").CowId)).reset_index(drop=True)
            M = MatchCalving(C, B, Pool, PBands[Named], Expected=Calved)
            M["Chance"] = round(len(Calved & set(Pool.CowId)) / Pool.CowId.nunique(), 3)
        if len(M):
            Out.append(M.assign(Folder=Name))
    return pd.concat(Out, ignore_index=True) if Out else pd.DataFrame()
if Ids is None:
    Tried = {L: Ranked(L) for L in ("A", "B")}
    Layout = max(Tried, key=lambda L: Tried[L].Margin.fillna(0).mean() if len(Tried[L]) else -1)
    Matches = Tried[Layout]
    print(f"Stalls 1-28 taken on the {'left' if Layout == 'A' else 'right'} of the picture "
          f"(mean margin {Tried[Layout].Margin.mean():.3f} vs {Tried['B' if Layout == 'A' else 'A'].Margin.mean():.3f})")
else:
    Matches = Ranked(None)
Matches.to_csv(NamesDir / "CalvingMatches.csv", index=False)
def Sum(G):
    Votes = G.groupby("BestCow").Crops.sum().sort_values(ascending=False)
    Out = {"Tracks": len(G), "Crops": G.Crops.sum(), "TopCow": Votes.index[0], "TopCowShare": round(Votes.iloc[0] / G.Crops.sum(), 2),
           "Margin": round(G.Margin.median(), 3), "TiedWith": G.SecondCow[G.Margin <= 0.001].mode().iloc[0] if (G.Margin <= 0.001).any() else None}
    if "CalvedInParlor" in G:
        Out["Chance"] = round(1 / G.CalvedInParlor.iloc[0], 2)
    else:
        Out.update(ExpectedShare=round((G.Crops * G.Expected).sum() / G.Crops.sum(), 2), Chance=G.Chance.iloc[0],
                   BestCalvedCow=G.groupby("BestExpected").Crops.sum().idxmax() if G.BestExpected.notna().any() else None,
                   CalvedCowRank=G.ExpectedRank.median())
    return pd.Series(Out)
Summary = Matches.groupby("Folder").apply(Sum)
Summary.to_csv(NamesDir / "CalvingSummary.csv")
display(Summary)
print("TopCow = the farm ID this pen cow most likely is. Trust it more when TopCowShare is high, Margin is above 0, and "
      "the same cow comes out in both windows of a video and in the other videos of that pen and day.")

## About this notebook

Code: `CalvingPen/CalvingPipeline/Reid/ParlorIds.py` (report, clock, naming, matching), plus `Segments.py` (tracking and crops), `Bands.py` (fingerprints) and `Identity.py` (`CropScores`).

**Outputs** in `ReidPipelineOutputs/<Run>/`:
- `Parlor/Clock.json`: when each parlor video starts and its milkings; `Parlor/<video>Milkings.jpg`: a picture from the middle of each milking.
- `Parlor/<video>S<n>/`: tracks, crops and fingerprints of milking *n*.
- `Calving/<video>Win<k>/`: the same for window *k* of a calving video. *Win* = a time window of the recording.
- `Names/ParlorIds.csv`: the parlor tracks named with a farm ID (Score, Margin). `Names/ParlorCandidates.csv` has every track's possible cows.
- `Names/Cows/<CowId>.jpg`: one row per milking, to check by eye.
- `Names/CalvingMatches.csv`: each calving-pen track's best parlor cow, the runner-up, the margin, and whether the best is one that calved then.

**Limits:**
- The parlor side of the picture that holds stalls 1–28 is found by trying both.
- Tracks are split at the middle of the picture.
- The calving pens of November have no pen outline yet, so cows in neighbouring pens can appear. Those should come out as cows that did not calve.